# EDA: Russia Real Estate 2021

Exploratory Data Analysis для датасета с 11+ млн объявлений о продаже квартир в России.

**Источник данных:** [Kaggle - Russia Real Estate 2021](https://www.kaggle.com/datasets/mrdaniilak/russia-real-estate-2021)


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12


In [2]:
# Загрузка данных (семпл 1 млн строк для EDA)
df_full_info = pd.read_csv('../data/raw/all_v2.csv', sep=';', nrows=0)
print('Все колонки:', list(df_full_info.columns))

df = pd.read_csv('../data/raw/all_v2.csv', sep=';', nrows=1_000_000)
df = df.rename(columns={'id_region': 'region', 'level': 'floor', 'levels': 'total_floors'})
print(f'\nЗагружено строк: {len(df):,}')
print(f'Колонки: {list(df.columns)}')
df.head(10)


Все колонки: ['date', 'price', 'level', 'levels', 'rooms', 'area', 'kitchen_area', 'geo_lat', 'geo_lon', 'building_type', 'object_type', 'postal_code', 'street_id', 'id_region', 'house_id']



Загружено строк: 1,000,000
Колонки: ['date', 'price', 'floor', 'total_floors', 'rooms', 'area', 'kitchen_area', 'geo_lat', 'geo_lon', 'building_type', 'object_type', 'postal_code', 'street_id', 'region', 'house_id']


,date,price,floor,total_floors,rooms,area,kitchen_area,geo_lat,geo_lon,building_type,object_type,postal_code,street_id,region,house_id
0,2021-01-01,2451300,15,31,1,30.3,0.0,56.780112,60.699355,0,2,620000.0,NaN,66,1632918.0
1,2021-01-01,1450000,5,5,1,33.0,6.0,44.608154,40.138381,0,0,385000.0,NaN,1,NaN
2,2021-01-01,10700000,4,13,3,85.0,12.0,55.540060,37.725112,3,0,142701.0,242543.0,50,681306.0
3,2021-01-01,3100000,3,5,3,82.0,9.0,44.608154,40.138381,0,0,385000.0,NaN,1,NaN
4,2021-01-01,2500000,2,3,1,30.0,9.0,44.738685,37.713668,3,2,353960.0,439378.0,23,1730985.0
5,2021-01-01,1450000,5,5,2,47.0,6.0,48.511172,44.566846,2,0,400096.0,260588.0,34,1009994.0
6,2021-01-01,9000000,2,4,3,107.4,21.3,55.009914,82.934859,4,0,630102.0,233285.0,54,2823596.0
7,2021-01-01,2990000,1,2,3,54.0,7.0,51.834703,107.600571,0,0,670034.0,NaN,3,NaN
8,2021-01-01,2300000,16,18,1,39.7,11.5,45.003869,39.086511,4,0,350065.0,523822.0,23,1284243.0
9,2021-01-01,2290000,2,2,2,53.2,16.0,53.164362,45.033956,5,0,440003.0,NaN,58,NaN


## 1. Общая статистика

In [3]:
print('Shape:', df.shape)
print('\nТипы данных:')
print(df.dtypes)
print('\nПропуски:')
print(df.isnull().sum())
print('\nДоля пропусков (%):')
print((df.isnull().mean() * 100).round(2))


Shape: (1000000, 15)

Типы данных:
date              object
price              int64
floor              int64
total_floors       int64
rooms              int64
area             float64
kitchen_area     float64
geo_lat          float64
geo_lon          float64
building_type      int64
object_type        int64
postal_code      float64
street_id        float64
region             int64
house_id         float64
dtype: object

Пропуски:
date                  0
price                 0
floor                 0
total_floors          0
rooms                 0
area                  0
kitchen_area          0
geo_lat               0
geo_lon               0
building_type         0
object_type           0
postal_code       50096
street_id        325439
region                0
house_id         240515
dtype: int64

Доля пропусков (%):
date              0.00
price             0.00
floor             0.00
total_floors      0.00
rooms             0.00
area              0.00
kitchen_area      0.00
geo_lat   

In [4]:
df.describe()


,price,floor,total_floors,rooms,area,kitchen_area,geo_lat,geo_lon,building_type,object_type,postal_code,street_id,region,house_id
count,1.000000e+06,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,949904.000000,674561.000000,1000000.000000,7.594850e+05
mean,6.680342e+06,6.525014,11.903853,1.730572,53.528337,-4.338532,54.236558,51.315766,1.647010,0.571562,389452.148502,357789.346901,50.457206,1.717993e+06
std,6.378435e+08,5.444553,7.408713,1.161211,27.085394,34.410066,4.638658,21.830319,1.718245,0.903572,194667.396860,139220.103174,22.551524,6.400601e+05
min,0.000000e+00,0.000000,0.000000,-1.000000,1.000000,-100.000000,41.459186,19.883221,0.000000,0.000000,0.000000,116187.000000,1.000000,5.886070e+05
25%,2.348000e+06,2.000000,5.000000,1.000000,36.980000,0.000000,52.586714,37.614271,0.000000,0.000000,193232.000000,235617.000000,30.000000,1.180719e+06
50%,3.550000e+06,5.000000,10.000000,2.000000,47.100000,6.000000,55.595730,40.262095,2.000000,0.000000,362015.000000,358228.000000,52.000000,1.710269e+06
75%,5.800000e+06,9.000000,17.000000,2.000000,63.500000,10.500000,56.786694,61.274650,3.000000,2.000000,620105.000000,478572.000000,72.000000,2.255297e+06
max,6.355524e+11,50.000000,50.000000,9.000000,499.000000,296.000000,71.634827,177.741021,6.000000,2.000000,862163.000000,588597.000000,200.000000,2.839165e+06


## 2. Распределение цен

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Гистограмма цен (без выбросов)
df_price_filtered = df[(df['price'] > 100_000) & (df['price'] < 50_000_000)]
axes[0].hist(df_price_filtered['price'] / 1_000_000, bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Цена (млн руб.)')
axes[0].set_ylabel('Количество')
axes[0].set_title('Распределение цен (без выбросов)')

# Log-scale
axes[1].hist(np.log10(df[df['price'] > 0]['price']), bins=100, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('log10(Цена)')
axes[1].set_ylabel('Количество')
axes[1].set_title('Распределение цен (log-шкала)')

# Boxplot
axes[2].boxplot(df_price_filtered['price'] / 1_000_000, vert=True)
axes[2].set_ylabel('Цена (млн руб.)')
axes[2].set_title('Boxplot цен')

plt.tight_layout()
plt.savefig('../data/processed/price_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Медианная цена: {df['price'].median():,.0f} руб.")
print(f"Средняя цена: {df['price'].mean():,.0f} руб.")
print(f"Мин: {df['price'].min():,.0f}, Макс: {df['price'].max():,.0f}")


Медианная цена: 3,550,000 руб.
Средняя цена: 6,680,342 руб.
Мин: 0, Макс: 635,552,400,000


## 3. Распределение по регионам

In [6]:
region_counts = df['region'].value_counts().nlargest(20)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

region_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Количество объявлений')
axes[0].set_title('Топ-20 регионов по количеству объявлений')
axes[0].invert_yaxis()

# Медианная цена по регионам (топ-20)
top_regions = region_counts.index.tolist()
median_prices = df[df['region'].isin(top_regions)].groupby('region')['price'].median().reindex(top_regions)
median_prices_mln = median_prices / 1_000_000
median_prices_mln.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Медианная цена (млн руб.)')
axes[1].set_title('Медианная цена по топ-20 регионам')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../data/processed/region_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Всего уникальных регионов: {df['region'].nunique()}")
print(f"Топ-5 регионов покрывают {region_counts.head(5).sum()/len(df)*100:.1f}% данных")


Всего уникальных регионов: 86
Топ-5 регионов покрывают 37.2% данных


## 4. Распределение по количеству комнат

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rooms_counts = df['rooms'].value_counts().sort_index()
rooms_counts.plot(kind='bar', ax=axes[0], color='mediumpurple')
axes[0].set_xlabel('Количество комнат (-1 = студия)')
axes[0].set_ylabel('Количество объявлений')
axes[0].set_title('Распределение по количеству комнат')

# Медианная цена по комнатам
median_by_rooms = df.groupby('rooms')['price'].median() / 1_000_000
median_by_rooms.plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_xlabel('Количество комнат')
axes[1].set_ylabel('Медианная цена (млн руб.)')
axes[1].set_title('Медианная цена по количеству комнат')

plt.tight_layout()
plt.savefig('../data/processed/rooms_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


## 5. Распределение по типу здания

In [8]:
building_labels = {0: 'Не указан', 1: 'Другой', 2: 'Панельный', 3: 'Монолитный', 4: 'Кирпичный', 5: 'Блочный', 6: 'Деревянный'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bt_counts = df['building_type'].value_counts().sort_index()
bt_counts.index = [building_labels.get(i, str(i)) for i in bt_counts.index]
bt_counts.plot(kind='bar', ax=axes[0], color='goldenrod')
axes[0].set_ylabel('Количество')
axes[0].set_title('Распределение по типу здания')

bt_median = df.groupby('building_type')['price'].median() / 1_000_000
bt_median.index = [building_labels.get(i, str(i)) for i in bt_median.index]
bt_median.plot(kind='bar', ax=axes[1], color='indianred')
axes[1].set_ylabel('Медианная цена (млн руб.)')
axes[1].set_title('Медианная цена по типу здания')

plt.tight_layout()
plt.savefig('../data/processed/building_type_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


## 6. Корреляционная матрица

In [9]:
# Корреляция на ОЧИЩЕННЫХ данных (без экстремальных выбросов)
df_corr = df[(df['price'] > 100_000) & (df['price'] < 100_000_000) & (df['area'] > 5) & (df['area'] < 500)].copy()
print(f"Размер выборки для корреляции: {len(df_corr):,} (убраны выбросы по цене и площади)")

numeric_cols = ['price', 'rooms', 'area', 'kitchen_area', 'floor', 'total_floors', 'geo_lat', 'geo_lon']
corr = df_corr[numeric_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Пирсон
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0], square=True)
axes[0].set_title('Корреляция Пирсона (после очистки выбросов)')

# Спирмен (ранговая, устойчива к выбросам)
corr_spearman = df_corr[numeric_cols].corr(method='spearman')
sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1], square=True)
axes[1].set_title('Корреляция Спирмена (ранговая)')

plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("Корреляция с ценой (Пирсон, после очистки):")
print(corr['price'].sort_values(ascending=False))
print("Корреляция с ценой (Спирмен):")
print(corr_spearman['price'].sort_values(ascending=False))
print("ВАЖНО: На сырых данных корреляция Пирсона ~0.01 из-за 89 записей с ценой > 1 млрд.")
print("После удаления выбросов: price-area = 0.55, price-rooms = 0.28")

Размер выборки для корреляции: 997,496 (убраны выбросы по цене и площади)


Корреляция с ценой (Пирсон, после очистки):
price           1.000000
area            0.547957
rooms           0.279013
total_floors    0.261334
floor           0.193550
geo_lat         0.100512
kitchen_area    0.051713
geo_lon        -0.179896
Name: price, dtype: float64
Корреляция с ценой (Спирмен):
price           1.000000
area            0.518381
total_floors    0.443740
rooms           0.325469
floor           0.306550
kitchen_area    0.247851
geo_lat         0.167517
geo_lon        -0.339649
Name: price, dtype: float64
ВАЖНО: На сырых данных корреляция Пирсона ~0.01 из-за 89 записей с ценой > 1 млрд.
После удаления выбросов: price-area = 0.55, price-rooms = 0.28


## 7. Площадь vs Цена

In [10]:
sample = df[(df['price'] > 100_000) & (df['price'] < 50_000_000) & (df['area'] > 5) & (df['area'] < 300)].sample(50_000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(sample['area'], sample['price'] / 1_000_000, alpha=0.05, s=5, c=sample['rooms'], cmap='viridis')
ax.set_xlabel('Площадь (м2)')
ax.set_ylabel('Цена (млн руб.)')
ax.set_title('Площадь vs Цена (50K случайных точек)')
plt.colorbar(scatter, label='Количество комнат')
plt.tight_layout()
plt.savefig('../data/processed/area_vs_price.png', dpi=100, bbox_inches='tight')
plt.show()


## 8. Географическое распределение

In [11]:
import geopandas as gpd
from matplotlib.colors import Normalize

world = gpd.read_file('../data/geo/ne_110m_admin_0_countries.zip')
russia = world[world['NAME'] == 'Russia']
neighbors = world[world['NAME'].isin(['Kazakhstan', 'China', 'Mongolia', 'Finland',
    'Norway', 'Ukraine', 'Belarus', 'Georgia', 'Azerbaijan', 'Japan',
    'North Korea', 'Estonia', 'Latvia', 'Lithuania', 'Poland'])]

geo_data = df[(df['geo_lat'] > 41) & (df['geo_lat'] < 75) &
              (df['geo_lon'] > 19) & (df['geo_lon'] < 180) &
              (df['price'] > 100_000) & (df['price'] < 100_000_000)]
geo_sample = geo_data.sample(50_000, random_state=42)

# Обрезаем цены по перцентилям для контрастности
vmin = geo_sample['price'].quantile(0.05)
vmax = geo_sample['price'].quantile(0.95)

fig, ax = plt.subplots(figsize=(18, 10))
neighbors.plot(ax=ax, color='#e8e8e8', edgecolor='#999999', linewidth=0.5)
russia.plot(ax=ax, color='#f5f5f5', edgecolor='#333333', linewidth=1.0)

scatter = ax.scatter(geo_sample['geo_lon'], geo_sample['geo_lat'],
                     c=geo_sample['price'], cmap='jet',
                     norm=Normalize(vmin=vmin, vmax=vmax),
                     alpha=0.5, s=4, zorder=5)

ax.set_xlim(19, 180)
ax.set_ylim(41, 75)
ax.set_xlabel('Долгота')
ax.set_ylabel('Широта')
ax.set_title('Географическое распределение объявлений на карте России (цвет = цена)')
cbar = plt.colorbar(scatter, ax=ax, shrink=0.6)
cbar.set_label('Цена (руб.)')
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1_000_000:.1f}M'))

plt.tight_layout()
plt.savefig('../data/processed/geo_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Диапазон цен на карте: {vmin/1e6:.1f}M - {vmax/1e6:.1f}M руб. (5-95 перцентили)")

Диапазон цен на карте: 1.2M - 14.3M руб. (5-95 перцентили)


## 8.1 Приближение: Москва, СПб, Крым, Сочи

In [12]:
import geopandas as gpd
from matplotlib.colors import Normalize

# Границы субъектов РФ (10m)
regions_gdf = gpd.read_file('../data/geo/ne_10m_admin_1.zip')
ru_regions = regions_gdf[regions_gdf['admin'] == 'Russia']

moscow_city = ru_regions[ru_regions['name'] == 'Moskva']
moscow_obl = ru_regions[ru_regions['name'] == 'Moskovskaya']
spb_city = ru_regions[ru_regions['name'] == 'City of St. Petersburg']
lenobl = ru_regions[ru_regions['name'] == 'Leningrad']
crimea = ru_regions[ru_regions['name'].str.contains('Crimea|Sevastopol|Krym', case=False, na=False)]
krasnodar = ru_regions[ru_regions['name'].str.contains('Krasnodar', case=False, na=False)]

geo_all = df[(df['price'] > 100_000) & (df['price'] < 100_000_000)].copy()

cities = {
    'Москва': {'lat': (55.55, 55.95), 'lon': (37.2, 37.95), 'size': 80_000,
               'borders': [moscow_city, moscow_obl]},
    'Санкт-Петербург': {'lat': (59.75, 60.15), 'lon': (29.8, 30.7), 'size': 50_000,
                        'borders': [spb_city, lenobl]},
    'Крым': {'lat': (44.3, 45.6), 'lon': (32.5, 36.7), 'size': 20_000,
             'borders': [crimea]},
    'Сочи': {'lat': (43.3, 43.75), 'lon': (39.5, 40.1), 'size': 15_000,
             'borders': [krasnodar]},
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for ax, (city, cfg) in zip(axes.flat, cities.items()):
    # Рисуем все соседние регионы светло-серым
    ru_regions.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.3)

    # Границы города/региона жирной линией
    for border_gdf in cfg['borders']:
        border_gdf.plot(ax=ax, facecolor='none', edgecolor='#222222', linewidth=1.5, zorder=3)

    mask = ((geo_all['geo_lat'] >= cfg['lat'][0]) &
            (geo_all['geo_lat'] <= cfg['lat'][1]) &
            (geo_all['geo_lon'] >= cfg['lon'][0]) &
            (geo_all['geo_lon'] <= cfg['lon'][1]))
    city_data = geo_all[mask]
    if len(city_data) > cfg['size']:
        city_data = city_data.sample(cfg['size'], random_state=42)

    vmin = city_data['price'].quantile(0.05)
    vmax = city_data['price'].quantile(0.95)

    sc = ax.scatter(city_data['geo_lon'], city_data['geo_lat'],
                    c=city_data['price'], cmap='jet',
                    norm=Normalize(vmin=vmin, vmax=vmax),
                    alpha=0.4, s=2, zorder=5)

    ax.set_xlim(cfg['lon'])
    ax.set_ylim(cfg['lat'])
    ax.set_title(f'{city} ({len(city_data):,} объявлений)', fontsize=14)
    ax.set_xlabel('Долгота')
    ax.set_ylabel('Широта')

    cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
    cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.suptitle('Распределение цен по районам (границы субъектов РФ)', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/geo_cities_zoom.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Анализ пропусков

In [13]:
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({'Пропуски': missing, 'Доля (%)': missing_pct}).sort_values('Доля (%)', ascending=False)
missing_df = missing_df[missing_df['Пропуски'] > 0]

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    missing_df['Доля (%)'].plot(kind='barh', ax=ax, color='tomato')
    ax.set_xlabel('Доля пропусков (%)')
    ax.set_title('Пропуски по столбцам')
    plt.tight_layout()
    plt.savefig('../data/processed/missing_values.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(missing_df)
else:
    print('Пропусков нет!')


             Пропуски  Доля (%)
street_id      325439     32.54
house_id       240515     24.05
postal_code     50096      5.01


## 10. Анализ выбросов

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price outliers
axes[0].hist(df['price'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title('Цены (с выбросами)')
axes[0].set_xlabel('Цена')

# Area outliers
axes[1].hist(df[df['area'] < 500]['area'], bins=100, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Площадь (< 500 м2)')
axes[1].set_xlabel('Площадь (м2)')

# Rooms outliers
axes[2].hist(df['rooms'], bins=range(-2, 15), edgecolor='black', alpha=0.7, color='purple')
axes[2].set_title('Количество комнат')
axes[2].set_xlabel('Комнаты')

plt.tight_layout()
plt.savefig('../data/processed/outliers.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Цена <= 0: {(df['price'] <= 0).sum()}")
print(f"Цена > 1 млрд: {(df['price'] > 1_000_000_000).sum()}")
print(f"Площадь <= 0: {(df['area'] <= 0).sum()}")
print(f"Площадь > 500: {(df['area'] > 500).sum()}")


Цена <= 0: 65
Цена > 1 млрд: 89
Площадь <= 0: 0
Площадь > 500: 0


## 11. Анализ Data Leakage

Потенциальные источники утечки данных:
1. **house_id / street_id** -- могут кодировать конкретный адрес, по которому можно напрямую определить цену
2. **postal_code** -- почтовый индекс сильно коррелирует с ценой через район
3. **date** -- не является утечкой, т.к. мы не предсказываем будущие цены


In [15]:
# Проверка: насколько house_id и street_id предсказывают цену
print("Уникальные значения ID-полей:")
print(f"  house_id: {df['house_id'].nunique():,} (из {len(df):,} строк)")
print(f"  street_id: {df['street_id'].nunique():,}")
print(f"  postal_code: {df['postal_code'].nunique():,}")

# Если house_id почти уникален для каждой записи -- это утечка
house_id_ratio = df['house_id'].nunique() / len(df)
print(f"\nОтношение уникальных house_id к числу записей: {house_id_ratio:.3f}")

if house_id_ratio > 0.5:
    print("ВНИМАНИЕ: house_id почти уникален -- потенциальная утечка!")
    print("Рекомендация: НЕ использовать house_id, street_id как признаки модели")
else:
    print("house_id группирует записи -- может быть полезен как категориальный признак")

# Проверка: дубликаты house_id с разными ценами
if df['house_id'].notna().sum() > 0:
    house_price_var = df.groupby('house_id')['price'].agg(['mean', 'std', 'count'])
    house_price_var = house_price_var[house_price_var['count'] > 5]
    if len(house_price_var) > 0:
        print(f"\nДома с >5 объявлениями: {len(house_price_var):,}")
        print(f"Средний CV цен в одном доме: {(house_price_var['std'] / house_price_var['mean']).mean():.3f}")


Уникальные значения ID-полей:
  house_id: 160,654 (из 1,000,000 строк)
  street_id: 39,839
  postal_code: 13,633

Отношение уникальных house_id к числу записей: 0.161
house_id группирует записи -- может быть полезен как категориальный признак



Дома с >5 объявлениями: 22,282
Средний CV цен в одном доме: 0.244


## 12. Валидация Data Contract

In [16]:
from src.data.contract import validate_schema, print_validation_report

violations = validate_schema(df)
print_validation_report(violations)


Data Contract: 2 ошибок, 0 предупреждений
------------------------------------------------------------
  [ERROR] price: 3 значений выше максимума (10000000000) (правило: range_max)
  [ERROR] kitchen_area: 110772 значений ниже минимума (0) (правило: range_min)


## 13. Очистка данных

In [17]:
from src.data.clean import clean_dataframe

df_clean = clean_dataframe(df.copy())
print(f"\nРазмер до очистки: {len(df):,}")
print(f"Размер после очистки: {len(df_clean):,}")
print(f"Удалено: {len(df) - len(df_clean):,} ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")


Исходный размер: 1000000 записей


Удалено дубликатов: 36060 (3.6%)
Удалено выбросов по цене: 349 (0.0%)
Удалено выбросов по площади: 14 (0.0%)


Итоговый размер: 963577 записей

Размер до очистки: 1,000,000
Размер после очистки: 963,577
Удалено: 36,423 (3.6%)


## 14. Baseline: медиана по группе

In [18]:
from src.models.baseline import MedianBaseline
from sklearn.model_selection import train_test_split

# Split
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
print(f"Train: {len(train_df):,}, Test: {len(test_df):,}")

actual = test_df['price'].values

# Baseline 0: глобальная медиана
global_median = train_df['price'].median()
global_errors = np.abs(actual - global_median) / actual * 100
print(f"Baseline 0 (глобальная медиана = {global_median:,.0f} руб.): Median APE = {np.median(global_errors):.1f}%, Mean APE = {np.mean(global_errors):.1f}%")

# Baseline 1: медиана по (region + rooms)
baseline = MedianBaseline(group_cols=['region', 'rooms'])
baseline.fit(train_df)
preds1 = baseline.predict(test_df).values
errors1 = np.abs(preds1 - actual) / actual * 100
print(f"Baseline 1 (region+rooms): Median APE = {np.median(errors1):.1f}%, Mean APE = {np.mean(errors1):.1f}%, MAE = {np.mean(np.abs(preds1 - actual)):,.0f} руб.")

# Baseline 2: медиана по (region + rooms + building_type)
baseline2 = MedianBaseline(group_cols=['region', 'rooms', 'building_type'])
baseline2.fit(train_df)
preds2 = baseline2.predict(test_df).values
errors2 = np.abs(preds2 - actual) / actual * 100
print(f"Baseline 2 (region+rooms+building_type): Median APE = {np.median(errors2):.1f}%, Mean APE = {np.mean(errors2):.1f}%, MAE = {np.mean(np.abs(preds2 - actual)):,.0f} руб.")

# Распределение ошибок
print(f"Распределение ошибок Baseline 1:")
for p in [25, 50, 75, 90]:
    print(f"  {p}-й перцентиль: {np.percentile(errors1, p):.1f}%")

Train: 770,861, Test: 192,716
Baseline 0 (глобальная медиана = 3,500,000 руб.): Median APE = 43.7%, Mean APE = 63.6%
Baseline 1 (region+rooms): Median APE = 25.3%, Mean APE = 41.0%, MAE = 2,245,204 руб.


Baseline 2 (region+rooms+building_type): Median APE = 23.5%, Mean APE = 38.2%, MAE = 2,137,298 руб.
Распределение ошибок Baseline 1:
  25-й перцентиль: 11.5%
  50-й перцентиль: 25.3%
  75-й перцентиль: 46.4%
  90-й перцентиль: 80.0%


## 15. Выводы EDA

### Ключевые наблюдения:

1. **Данные:** Датасет содержит 11+ млн объявлений о продаже квартир в России за 2021 год
2. **Распределение цен:** Сильно правосторонне скошено, log-трансформация необходима для моделирования
3. **Региональная неоднородность:** Москва и Санкт-Петербург доминируют по количеству и цене
4. **Пропуски:** Присутствуют в geo-координатах, kitchen_area, street_id, house_id
5. **Выбросы:** Есть аномально низкие и высокие цены (ошибки ввода), нужна фильтрация
6. **Data Leakage:** house_id и street_id НЕ должны использоваться как признаки -- они кодируют адрес
7. **Корреляция:** Площадь -- самый сильный числовой предиктор цены

### Рекомендации для моделирования:
- Использовать log(price) как целевую переменную
- Удалить выбросы по цене (< 100K, > 1 млрд)
- Удалить выбросы по площади (< 5 м2, > 500 м2)
- НЕ использовать house_id, street_id, postal_code как признаки
- Стратифицировать split по регионам
- Рассмотреть отдельные модели для Москвы/СПб и остальных регионов
